# DESI DR2 BGS, Legacy Survey DR9, and redMaPPer Footprints

This notebook makes a common-sky Mollweide diagnostic using:

- the DESI DR2 BGS Bright random catalog to trace the BGS angular selection footprint;
- the Legacy Survey DR9 random catalog as a muted imaging-coverage background; and
- unique redMaPPer BCG positions from `RM_SDSS_df.pkl`.

Random catalogs trace angular selection and coverage. They should not be interpreted as the observed galaxy overdensity field. The FITS files are processed in chunks so the notebook does not load an entire random catalog into memory.

In [ ]:
from pathlib import Path
import pickle
import warnings

import healpy as hp
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, Normalize, to_rgba
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
from astropy.io import fits
from astropy.table import Table, unique
from astropy.units import UnitsWarning
from IPython.display import display

warnings.filterwarnings('ignore', category=UnitsWarning)

mpl.rcParams.update({
    'font.size': 12,
    'axes.linewidth': 1.1,
    'xtick.direction': 'in',
    'ytick.direction': 'in',
})

## Configuration

In [ ]:
REPO_ROOT = Path('/global/homes/z/zzhang13/DESI/Projection')
if not REPO_ROOT.exists():
    REPO_ROOT = Path.cwd().resolve()
    if REPO_ROOT.name in {'local_overdensity', 'make_catalogs', 'richness_relation', 'plotting'}:
        REPO_ROOT = REPO_ROOT.parent

CATALOG_DIR = REPO_ROOT / 'catalogs'
RM_CATALOG = CATALOG_DIR / 'RM_SDSS_df.pkl'

DR2_DIR = Path('/global/cfs/cdirs/desi/survey/catalogs/DA2/LSS/loa-v1/LSScats/v2.1')
DR2_RANDOM_PATTERN = 'BGS_BRIGHT_*_full.ran.fits'

DR9_RANDOM_FILE = Path(
    '/global/cfs/cdirs/desi/target/catalogs/dr9/0.49.0/'
    'randoms/resolve/randoms-10-0.fits'
)

OUTPUT_DIR = REPO_ROOT / 'plots'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_BASENAME = 'dr2_bgs_dr9_rm_footprint'

NSIDE = 128
CHUNK_ROWS = 2_000_000

# One DR2 random realization is normally dense enough for a binary footprint.
# Set to None to combine every available realization.
MAX_DR2_RANDOM_FILES = 1

dr2_random_files = sorted(DR2_DIR.glob(DR2_RANDOM_PATTERN))
if MAX_DR2_RANDOM_FILES is not None:
    dr2_random_files = dr2_random_files[:MAX_DR2_RANDOM_FILES]

if not RM_CATALOG.exists():
    raise FileNotFoundError(f'Missing redMaPPer catalog: {RM_CATALOG}')
if not dr2_random_files:
    raise FileNotFoundError(f'No DR2 random catalogs match {DR2_DIR / DR2_RANDOM_PATTERN}')
if not DR9_RANDOM_FILE.exists():
    raise FileNotFoundError(f'Missing Legacy Survey DR9 random catalog: {DR9_RANDOM_FILE}')

print('Repository:', REPO_ROOT)
print('redMaPPer catalog:', RM_CATALOG)
print(f'DR2 random files selected: {len(dr2_random_files):,}')
for path in dr2_random_files:
    print('  ', path.name)
print('DR9 random catalog:', DR9_RANDOM_FILE)
print('Output directory:', OUTPUT_DIR)

## Catalog and HEALPix helpers

In [ ]:
def read_pickle_table(path):
    with Path(path).open('rb') as handle:
        obj = pickle.load(handle)
    if isinstance(obj, Table):
        return obj
    if obj.__class__.__module__.startswith('pandas'):
        return Table.from_pandas(obj)
    if hasattr(obj, 'to_pandas'):
        return Table.from_pandas(obj.to_pandas())
    return Table(obj)


def first_column(table, *candidates):
    for column in candidates:
        if column in table.colnames:
            return column
    raise KeyError(f'None of these columns is present: {candidates}')


def fits_table_hdu(hdul):
    for hdu in hdul[1:]:
        if isinstance(hdu, (fits.BinTableHDU, fits.TableHDU)) and hdu.data is not None:
            names = set(hdu.columns.names or [])
            if {'RA', 'DEC'}.issubset(names):
                return hdu
    raise KeyError('No FITS table extension contains both RA and DEC.')


def accumulate_healpix_counts(paths, nside=128, chunk_rows=2_000_000, label='catalog'):
    paths = [Path(path) for path in paths]
    counts = np.zeros(hp.nside2npix(nside), dtype=np.int64)
    total_rows = 0

    for file_number, path in enumerate(paths, start=1):
        print(f'{label}: reading {file_number}/{len(paths)}: {path.name}', flush=True)
        with fits.open(path, memmap=True) as hdul:
            data = fits_table_hdu(hdul).data
            n_rows = len(data)
            for start in range(0, n_rows, chunk_rows):
                stop = min(start + chunk_rows, n_rows)
                ra = np.asarray(data['RA'][start:stop], dtype=float)
                dec = np.asarray(data['DEC'][start:stop], dtype=float)
                valid = (
                    np.isfinite(ra)
                    & np.isfinite(dec)
                    & (dec >= -90.0)
                    & (dec <= 90.0)
                )
                if not np.any(valid):
                    continue
                pixels = hp.ang2pix(nside, ra[valid] % 360.0, dec[valid], lonlat=True)
                counts += np.bincount(pixels, minlength=len(counts))
                total_rows += np.count_nonzero(valid)

        print(f'  accumulated valid rows: {total_rows:,}', flush=True)

    return counts, total_rows

## Load unique redMaPPer cluster centers

In [ ]:
rm_all = read_pickle_table(RM_CATALOG)
ra_column = first_column(rm_all, 'RA_central', 'RA_x', 'RA')
dec_column = first_column(rm_all, 'DEC_central', 'DEC_x', 'DEC')

center_columns = ['ID', ra_column, dec_column]
if 'Z_SPEC_central' in rm_all.colnames:
    center_columns.append('Z_SPEC_central')
elif 'Z_SPEC_x' in rm_all.colnames:
    center_columns.append('Z_SPEC_x')

rm_centers = unique(rm_all[center_columns], keys='ID')
ra_rm = np.asarray(rm_centers[ra_column], dtype=float)
dec_rm = np.asarray(rm_centers[dec_column], dtype=float)
valid_rm = (
    np.isfinite(ra_rm)
    & np.isfinite(dec_rm)
    & (dec_rm >= -90.0)
    & (dec_rm <= 90.0)
)
ra_rm = ra_rm[valid_rm] % 360.0
dec_rm = dec_rm[valid_rm]

print(f'Input redMaPPer rows: {len(rm_all):,}')
print(f'Unique valid cluster centers: {len(ra_rm):,}')

## Build the DR2 and DR9 maps

The DR2 map is converted to a binary footprint for display. The DR9 map is shown as random-point surface density in units of deg$^{-2}$, which visualizes imaging coverage and masks.

In [ ]:
dr2_counts, n_dr2_randoms = accumulate_healpix_counts(
    dr2_random_files,
    nside=NSIDE,
    chunk_rows=CHUNK_ROWS,
    label='DESI DR2 BGS randoms',
)

dr9_counts, n_dr9_randoms = accumulate_healpix_counts(
    [DR9_RANDOM_FILE],
    nside=NSIDE,
    chunk_rows=CHUNK_ROWS,
    label='Legacy Survey DR9 randoms',
)

pixel_area_deg2 = hp.nside2pixarea(NSIDE, degrees=True)
dr2_footprint = dr2_counts > 0
dr9_footprint = dr9_counts > 0
dr9_density_deg2 = dr9_counts.astype(float) / pixel_area_deg2
dr9_density_deg2[~dr9_footprint] = np.nan

print(f'HEALPix NSIDE: {NSIDE}')
print(f'Pixel area: {pixel_area_deg2:.4f} deg^2')
print(f'DR2 random rows used: {n_dr2_randoms:,}')
print(f'DR9 random rows used: {n_dr9_randoms:,}')
print(f'DR2 occupied pixels: {np.count_nonzero(dr2_footprint):,}')
print(f'DR9 occupied pixels: {np.count_nonzero(dr9_footprint):,}')

## Cluster-footprint overlap

In [ ]:
rm_pixels = hp.ang2pix(NSIDE, ra_rm, dec_rm, lonlat=True)
inside_dr2 = dr2_footprint[rm_pixels]
inside_dr9 = dr9_footprint[rm_pixels]
inside_both = inside_dr2 & inside_dr9

overlap_summary = pd.DataFrame([
    {
        'selection': 'inside Legacy Survey DR9 random footprint',
        'N_clusters': int(np.count_nonzero(inside_dr9)),
        'fraction': np.mean(inside_dr9),
    },
    {
        'selection': 'inside DESI DR2 BGS random footprint',
        'N_clusters': int(np.count_nonzero(inside_dr2)),
        'fraction': np.mean(inside_dr2),
    },
    {
        'selection': 'inside both footprints',
        'N_clusters': int(np.count_nonzero(inside_both)),
        'fraction': np.mean(inside_both),
    },
])
display(overlap_summary)

## Mollweide overlay

In [ ]:
def map_on_mollweide_grid(healpix_map, n_lon=720, n_lat=360):
    x_edges = np.linspace(-np.pi, np.pi, n_lon + 1)
    y_edges = np.linspace(-0.5 * np.pi, 0.5 * np.pi, n_lat + 1)
    x_centers = 0.5 * (x_edges[:-1] + x_edges[1:])
    y_centers = 0.5 * (y_edges[:-1] + y_edges[1:])
    x_grid, y_grid = np.meshgrid(x_centers, y_centers)

    # Reverse longitude so right ascension increases to the left.
    ra_grid = (-np.rad2deg(x_grid)) % 360.0
    dec_grid = np.rad2deg(y_grid)
    pixels = hp.ang2pix(hp.npix2nside(len(healpix_map)), ra_grid, dec_grid, lonlat=True)
    image = np.asarray(healpix_map)[pixels]
    return x_edges, y_edges, image


def wrapped_ra_radians(ra_deg):
    ra_deg = np.asarray(ra_deg, dtype=float)
    return -np.deg2rad((ra_deg + 180.0) % 360.0 - 180.0)


x_edges, y_edges, dr9_image = map_on_mollweide_grid(dr9_density_deg2)
_, _, dr2_image = map_on_mollweide_grid(dr2_footprint.astype(float))

positive_dr9 = dr9_density_deg2[np.isfinite(dr9_density_deg2) & (dr9_density_deg2 > 0)]
vmin, vmax = np.nanpercentile(positive_dr9, [2, 98])
density_norm = Normalize(vmin=vmin, vmax=vmax, clip=True)

fig = plt.figure(figsize=(12.0, 7.2))
ax = fig.add_subplot(111, projection='mollweide')
fig.subplots_adjust(left=0.04, right=0.98, top=0.90, bottom=0.20)

dr9_mesh = ax.pcolormesh(
    x_edges,
    y_edges,
    np.ma.masked_invalid(dr9_image),
    cmap='Greys',
    norm=density_norm,
    shading='auto',
    rasterized=True,
    zorder=0,
)

dr2_overlay = np.ma.masked_where(dr2_image <= 0, dr2_image)
dr2_cmap = ListedColormap([to_rgba('#3b82a0', 0.38)])
ax.pcolormesh(
    x_edges,
    y_edges,
    dr2_overlay,
    cmap=dr2_cmap,
    vmin=0,
    vmax=1,
    shading='auto',
    rasterized=True,
    zorder=1,
)

ax.scatter(
    wrapped_ra_radians(ra_rm),
    np.deg2rad(dec_rm),
    s=7,
    alpha=0.72,
    color='#b2182b',
    edgecolors='none',
    rasterized=True,
    zorder=3,
)

if np.any(~inside_dr2):
    ax.scatter(
        wrapped_ra_radians(ra_rm[~inside_dr2]),
        np.deg2rad(dec_rm[~inside_dr2]),
        s=28,
        facecolors='none',
        edgecolors='#f0b429',
        linewidths=1.0,
        rasterized=True,
        zorder=4,
    )

ax.grid(color='white', alpha=0.32, lw=0.6)
ax.set_title('DESI DR2 BGS footprint and redMaPPer centers within Legacy Survey DR9', pad=18)

ra_tick_positions = np.deg2rad(np.arange(-150, 180, 30))
ra_tick_labels = ['150', '120', '90', '60', '30', '0', '330', '300', '270', '240', '210']
ax.set_xticks(ra_tick_positions)
ax.set_xticklabels([rf'${label}^\circ$' for label in ra_tick_labels])

legend_handles = [
    Line2D([0], [0], color='#3b82a0', lw=7, alpha=0.55, label='DESI DR2 BGS footprint'),
    Line2D(
        [0], [0], marker='o', color='none', markerfacecolor='#b2182b',
        markeredgecolor='none', markersize=6, label='redMaPPer BCG centers'
    ),
]
if np.any(~inside_dr2):
    legend_handles.append(
        Line2D(
            [0], [0], marker='o', color='none', markerfacecolor='none',
            markeredgecolor='#f0b429', markersize=7, label='outside DR2 BGS footprint'
        )
    )

fig.legend(
    handles=legend_handles,
    loc='lower left',
    bbox_to_anchor=(0.055, 0.035),
    frameon=False,
    ncol=1,
)

colorbar_axis = fig.add_axes([0.42, 0.075, 0.48, 0.026])
colorbar = fig.colorbar(dr9_mesh, cax=colorbar_axis, orientation='horizontal')
colorbar.set_label(r'Legacy Survey DR9 random density [deg$^{-2}$]', labelpad=5)

png_path = OUTPUT_DIR / f'{OUTPUT_BASENAME}.png'
pdf_path = OUTPUT_DIR / f'{OUTPUT_BASENAME}.pdf'
fig.savefig(png_path, dpi=220, bbox_inches='tight')
fig.savefig(pdf_path, bbox_inches='tight')
plt.show()

print('Saved:', png_path)
print('Saved:', pdf_path)

## Interpretation

- The grayscale background is Legacy Survey DR9 random-point density and therefore traces imaging coverage and masks.
- The blue layer is the binary DESI DR2 BGS Bright random footprint.
- Red points are unique redMaPPer BCG centers.
- Gold open circles, when present, identify cluster centers outside the occupied DR2 BGS HEALPix footprint.

For footprint membership near survey edges, repeat the map at a larger `NSIDE` and combine all DR2 random realizations by setting `MAX_DR2_RANDOM_FILES = None`.